# Phase 5 — Resolution Sensitivity Analysis

Measures how VGGT's pose accuracy and reconstruction quality depend on input
image resolution. VGGT was trained at **518 px** square inputs. This notebook
evaluates whether lower resolutions (cheaper, faster) are still accurate enough.

**Experiment**
1. Download TUM-VI `room1` (40 frames) with ground-truth poses.
2. Run VGGT at resolutions: 224, 280, 336, 392, 448, 518 px.
3. Record ATE, RPE, rotation error, inference time, and peak GPU memory.
4. Plot error-vs-resolution curves.

## 0 — Setup

In [ ]:
import subprocess, sys, os

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *args])

def git(*args):
    subprocess.check_call(["git"] + list(args))

pip("--upgrade", "numpy")
pip("plotly", "pandas", "pillow", "scipy", "matplotlib")

if not os.path.isdir("vggt"):
    git("clone", "--depth", "1", "https://github.com/facebookresearch/vggt.git")
    pip("-e", "vggt")

if not os.path.isdir("vggt-eval"):
    git("clone", "--depth", "1", "https://github.com/insha-py/vggt-eval.git")
else:
    git("-C", "vggt-eval", "pull", "--ff-only")

sys.path.insert(0, "vggt-eval")
sys.path.insert(0, "vggt")

print("Packages installed. Restarting kernel ...")
try:
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)
except Exception:
    print("Restart the kernel manually, then run from the Imports cell.")

In [ ]:
import os, sys
for p in ["vggt-eval", "vggt"]:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.tum_vi            import TUMVIDataset
from src.resolution_sweep  import ResolutionSweeper, DEFAULT_RESOLUTIONS
from src.metrics           import compute_ate, compute_rpe, print_metrics_table

np.random.seed(42)
torch.manual_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 1 — Download TUM-VI `room1`

In [ ]:
SEQUENCE     = "room1"
N_FRAMES     = 40
DOWNLOAD_DIR = "/tmp/tumvi"

ds   = TUMVIDataset(sequence=SEQUENCE, n_frames=N_FRAMES, download_dir=DOWNLOAD_DIR)
ds.download()
data = ds.load()

image_dir        = data["image_dir"]
image_timestamps = data["image_timestamps"]
gt_extrinsics    = data["gt_extrinsics"]   # (N, 3, 4)

N = len(gt_extrinsics)
print(f"Loaded {N} frames   GT shape: {gt_extrinsics.shape}")


## 2 — Load Model

In [ ]:
# chunk_size=8, overlap=3: sliding-window mode
# Allows 40 frames at all resolutions without OOM on T4 16 GB.
# Procrustes alignment stitches chunks; expect slightly higher ATE
# than a single-pass run on the same frames.
sweeper = ResolutionSweeper(
    resolutions      = DEFAULT_RESOLUTIONS,   # [224, 280, 336, 392, 448, 518]
    conf_thresh      = 5.0,
    store_extrinsics = False,
    chunk_size       = 8,    # frames per VGGT call
    overlap          = 3,    # overlap for Procrustes alignment
)
sweeper.load_model()


## 3 — Run Resolution Sweep

In [ ]:
results = sweeper.run(
    image_dir     = image_dir,
    max_frames    = N_FRAMES,
    gt_extrinsics = gt_extrinsics,
    align         = True,
    with_scale    = True,
)

df = sweeper.to_dataframe(results)
print("\nResolution sweep results:")
print(df.to_string(index=False, float_format="{:.4f}".format))

## 4 — Time & Memory vs Resolution

In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Inference Time (s)", "Peak GPU Memory (MB)"))

fig.add_trace(
    go.Scatter(x=df["resolution"], y=df["time_s"],
               mode="lines+markers", name="Time (s)",
               line=dict(color="steelblue", width=2), marker=dict(size=8)),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(x=df["resolution"], y=df["peak_mb"],
               mode="lines+markers", name="Peak MB",
               line=dict(color="darkorange", width=2), marker=dict(size=8)),
    row=1, col=2,
)
fig.update_xaxes(title_text="Resolution (px)")
fig.update_layout(height=400, title_text="Compute cost vs Input Resolution", showlegend=False)
fig.show()

## 5 — ATE vs Resolution

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df["resolution"], y=df["ate_mean"],
    mode="lines+markers", name="ATE mean",
    line=dict(color="royalblue", width=2), marker=dict(size=8),
))
fig.add_trace(go.Scatter(
    x=df["resolution"], y=df["ate_rmse"],
    mode="lines+markers", name="ATE RMSE",
    line=dict(color="royalblue", width=2, dash="dash"), marker=dict(size=6),
))
fig.add_trace(go.Scatter(
    x=df["resolution"], y=df["ate_median"],
    mode="lines+markers", name="ATE median",
    line=dict(color="royalblue", width=2, dash="dot"), marker=dict(size=6),
))

native_ate = df.loc[df["resolution"] == 518, "ate_mean"].values
if len(native_ate):
    fig.add_vline(x=518, line_dash="dash", line_color="grey",
                  annotation_text="Native 518 px", annotation_position="top left")

fig.update_layout(
    title=f"ATE vs Input Resolution -- TUM-VI {SEQUENCE} ({N} frames)",
    xaxis_title="Resolution (px)",
    yaxis_title="ATE (m, Umeyama-aligned)",
    legend=dict(x=0.6, y=0.95),
)
fig.show()

## 6 — RPE vs Resolution

In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("RPE Translation (m)", "RPE Rotation (deg)"))

fig.add_trace(
    go.Scatter(x=df["resolution"], y=df["rpe_trans"],
               mode="lines+markers", line=dict(color="forestgreen", width=2),
               marker=dict(size=8), name="RPE trans"),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(x=df["resolution"], y=df["rpe_rot"],
               mode="lines+markers", line=dict(color="firebrick", width=2),
               marker=dict(size=8), name="RPE rot"),
    row=1, col=2,
)
fig.update_xaxes(title_text="Resolution (px)")
fig.update_layout(height=400, title_text="Relative Pose Error vs Resolution", showlegend=False)
fig.show()

## 7 — Rotation Error vs Resolution

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["resolution"], y=df["rot_mean_deg"],
    mode="lines+markers", name="Mean rotation error (deg)",
    line=dict(color="purple", width=2), marker=dict(size=8),
))
fig.update_layout(
    title="Mean Rotation Error vs Input Resolution",
    xaxis_title="Resolution (px)",
    yaxis_title="Mean rotation error (deg)",
)
fig.show()

## 8 — Depth Confidence vs Resolution

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["resolution"], y=df["depth_conf_mean"],
    mode="lines+markers", name="Mean depth confidence",
    line=dict(color="teal", width=2), marker=dict(size=8),
))
fig.update_layout(
    title="Mean Depth Confidence vs Resolution",
    xaxis_title="Resolution (px)",
    yaxis_title="Mean confidence (0-1)",
)
fig.show()

## 9 — Accuracy / Efficiency Trade-off

In [ ]:
fig = px.scatter(
    df, x="time_s", y="ate_mean",
    text="resolution", color="resolution",
    color_continuous_scale="Viridis",
    title="Accuracy-Efficiency Trade-off (lower-left = better)",
    labels={"time_s": "Inference time (s)", "ate_mean": "ATE mean (m)"},
    size=[10] * len(df),
)
fig.update_traces(textposition="top center")
fig.show()

df["ate_per_sec"] = df["ate_mean"] / df["time_s"]
best = df.loc[df["ate_per_sec"].idxmin()]
print(f"Best accuracy-per-second: {int(best['resolution'])} px  "
      f"(ATE/s = {best['ate_per_sec']:.4f})")

## 10 — Summary Table & Save

In [ ]:
display_cols = [
    "resolution", "ate_mean", "ate_rmse",
    "rpe_trans", "rpe_rot", "rot_mean_deg",
    "time_s", "peak_mb",
]

print(f"\n=== Resolution Sensitivity -- TUM-VI {SEQUENCE} ({N} frames) ===")
print(df[display_cols].to_string(index=False, float_format="{:.4f}".format))

os.makedirs("results", exist_ok=True)
df.to_csv(f"results/phase5_resolution_sweep_{SEQUENCE}.csv", index=False)
print("\nSaved to results/phase5_resolution_sweep_room1.csv")

## Discussion

### Why does ATE increase at low resolution?

VGGT uses a Vision Transformer with patch-based attention. At lower resolutions:
- Fewer patches -> less positional context for pose estimation.
- Fine texture cues are lost.
- Depth estimation degrades.

### Connection to thesis

This motivates:
1. **Adaptive resolution** (Notebook 05): global 224 px + patch-refine uncertain regions at 518 px.
2. **Progressive refinement** (Notebook 05): coarse-to-fine pyramid blending depth across levels.